# NLP

## Регулярные выражения

https://tproger.ru/translations/regular-expression-python

Грубо говоря, у нас есть input-поле, в которое должен вводиться email-адрес. Но пока мы не зададим проверку валидности введённого email-адреса, в этой строке может оказаться совершенно любой набор символов, а нам это не нужно.

Чтобы выявить ошибку при вводе некорректного адреса электронной почты, можно использовать следующее регулярное выражение:

r'^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+(?:\.[a-zA-Z0-9-]+)+$'

In [1]:
import re

# match
ищет по заданному шаблону в начале строки

In [2]:
result = re.match('ab+c.', 'abcdefghijkabcabc') # ищем по шаблону 'ab+c.' 
print (result) # совпадение найдено:

<re.Match object; span=(0, 4), match='abcd'>


In [3]:
print(result.group(0)) # выводим найденное совпадение

abcd


In [4]:
result = re.match('abc.', 'abdefghijkabcabc')
print(result) # совпадение не найдено

None


In [5]:
result = re.match(r'AV An', 'AV Analytics Vidhya AV')
print(result)

<re.Match object; span=(0, 5), match='AV An'>


In [6]:
print(result.group(0))

AV An


# search
ищет по всей строке, возвращает только первое найденное совпадение

In [7]:
result = re.search('ch+i...', 'aefgabchijkabcabc') 
print(result) 

<re.Match object; span=(6, 12), match='chijka'>


In [8]:
print(result.group(0)) # выводим найденное совпадение

chijka


# findall
возвращает список всех найденных совпадений

In [9]:
result = re.findall('ab+c.', 'abcdefghijkabcaabcxabc') 
print(result)

['abcd', 'abca', 'abcx']


Вопросы: 
почему нет последнего abc?

# split
разделяет строку по заданному шаблону


In [11]:
t = 'itsy, bitsy, teenie, weenie'

In [15]:
t

'itsy, bitsy, teenie, weenie'

In [10]:
result = re.split(',', 'itsy, bitsy, teenie, weenie') 
print(result)

['itsy', ' bitsy', ' teenie', ' weenie']


In [13]:
result[3]

' weenie'

можно указать максимальное количество разбиений

In [16]:
result = re.split(',', 'itsy, bitsy, teenie, weenie', maxsplit = 2) 
print(result)

['itsy', ' bitsy', ' teenie, weenie']


# sub
ищет шаблон в строке и заменяет все совпадения на указанную подстроку

параметры: (pattern, repl, string)

In [17]:
result = re.sub('a', 'b', 'abcabc')
print (result)

bbcbbc


# compile
компилирует регулярное выражение в отдельный объект

In [18]:
# Пример: построение списка всех слов строки:
prog = re.compile('[А-Яа-яё\-]+')
prog.findall("Слова? Да, больше, ещё больше слов! Что-то ещё.")

['Слова', 'Да', 'больше', 'ещё', 'больше', 'слов', 'Что-то', 'ещё']

### Задача: классификация твитов по тональности

У нас есть датасет из твитов, про каждый указано, как он эмоционально окрашен: положительно или отрицательно. Задача: предсказывать эмоциональную окраску.

Классификацию по тональности используют в рекомендательных системах, чтобы понять, понравилось ли людям кафе, кино, etc.

Скачиваем куски датасета ([источник](http://study.mokoron.com/)): [положительные](https://www.dropbox.com/s/fnpq3z4bcnoktiv/positive.csv?dl=0), [отрицательные](https://www.dropbox.com/s/r6u59ljhhjdg6j0/negative.csv).

In [ ]:
# если у вас линукс / мак / collab или ещё какая-то среда, в которой работает wget, можно так:
#wget https://www.dropbox.com/s/fnpq3z4bcnoktiv/positive.csv
#!wget https://www.dropbox.com/s/r6u59ljhhjdg6j0/negative.csv

In [19]:
import pandas as pd
import numpy as np
from sklearn.metrics import *
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

In [20]:
# считываем данные и заполняем общий датасет
positive = pd.read_csv('positive.csv', sep=';', usecols=[3], names=['text'])
positive['label'] = ['positive'] * len(positive)
negative = pd.read_csv('negative.csv', sep=';', usecols=[3], names=['text'])
negative['label'] = ['negative'] * len(negative)
df = pd.concat([negative, positive])

In [21]:
df.shape

(226834, 2)

In [22]:
df.tail()

,text,label
114906,"Спала в родительском доме, на своей кровати......",positive
114907,RT @jebesilofyt: Эх... Мы немного решили сокра...,positive
114908,"Что происходит со мной, когда в эфире #proacti...",positive
114909,"""Любимая,я подарю тебе эту звезду..."" Имя како...",positive
114910,@Ma_che_rie посмотри #непытайтесьпокинутьомск ...,positive


In [23]:
df.sample(5)

,text,label
66223,"Мне так жалко стариков, которые этот новый год...",negative
4621,"@Saturnych в четверг в час ночи уже были в КЛ,...",positive
43800,"@fuzekrec жаль не попал, узнал за день до собы...",negative
73993,"Вчера своим бабам видеопоздравление отправила,...",negative
91841,"@Brigans ну я уже чейто чувствую, так что тута...",positive


In [24]:
df.loc[55490]['text'].iloc[1]

'Мороженое,нутела и фильм)больше ничего и не надо)))'

In [26]:
df.loc[5166]['text'].iloc[1]

'Урааа уже сегодня на каток как я ждала этого дня)))))'

In [27]:
x_train, x_test, y_train, y_test = train_test_split(df.text, df.label)

## Baseline: классификация необработанных n-грамм

### Векторизаторы

In [28]:
from sklearn.linear_model import LogisticRegression # можно заменить на любимый классификатор
from sklearn.feature_extraction.text import CountVectorizer

Что такое n-граммы:

In [29]:
from nltk import ngrams

In [30]:
sent = 'Если б мне платили каждый раз'.split()
list(ngrams(sent, 1)) # униграммы

[('Если',), ('б',), ('мне',), ('платили',), ('каждый',), ('раз',)]

In [31]:
list(ngrams(sent, 2)) # биграммы

[('Если', 'б'),
 ('б', 'мне'),
 ('мне', 'платили'),
 ('платили', 'каждый'),
 ('каждый', 'раз')]

In [32]:
list(ngrams(sent, 3)) # триграммы

[('Если', 'б', 'мне'),
 ('б', 'мне', 'платили'),
 ('мне', 'платили', 'каждый'),
 ('платили', 'каждый', 'раз')]

In [33]:
list(ngrams(sent, 5)) # ... пентаграммы?

[('Если', 'б', 'мне', 'платили', 'каждый'),
 ('б', 'мне', 'платили', 'каждый', 'раз')]

Самый простой способ извлечь фичи из текстовых данных -- векторизаторы: `CountVectorizer` и `TfidfVectorizer`

Объект `CountVectorizer` делает простую вещь:
* строит для каждого документа (каждой пришедшей ему строки) вектор размерности `n`, где `n` -- количество слов или n-грам во всём корпусе
* заполняет каждый i-тый элемент количеством вхождений слова в данный документ

In [34]:
vec = CountVectorizer(ngram_range = (1, 1))
bow = vec.fit_transform(x_train) # bow -- bag of words (мешок слов)

In [35]:
x_train[1000]

1000    Дим, ты помогаешь мне, я тебе, все взаимно, вс...
1000         И как я встану сегодня утром на Тренеровку:(
Name: text, dtype: object

In [36]:
bow

<170125x243842 sparse matrix of type '<class 'numpy.int64'>'
	with 1848126 stored elements in Compressed Sparse Row format>

ngram_range отвечает за то, какие n-граммы мы используем в качестве фичей:<br/>
ngram_range=(1, 1) -- униграммы<br/>
ngram_range=(3, 3) -- триграммы<br/>
ngram_range=(1, 3) -- униграммы, биграммы и триграммы.

В vec.vocabulary_ лежит словарь: мэппинг слов к их индексам:

In [37]:
list(vec.vocabulary_.items())[:10]

[('bugagagash', 19231),
 ('банально', 103362),
 ('одеваются', 172379),
 ('черно', 236777),
 ('серое', 208764),
 ('все', 115366),
 ('rt', 74571),
 ('origami_suvd', 66954),
 ('идэш', 139263),
 ('ирсэн', 141193)]

In [38]:
clf = LogisticRegression(random_state = 42)
clf.fit(bow, y_train)

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(random_state=42)

In [39]:
pred = clf.predict(vec.transform(x_test))
print(classification_report(pred, y_test))

              precision    recall  f1-score   support

    negative       0.77      0.76      0.76     28198
    positive       0.76      0.77      0.77     28511

    accuracy                           0.76     56709
   macro avg       0.76      0.76      0.76     56709
weighted avg       0.76      0.76      0.76     56709



Попробуем сделать то же самое для триграмм:

In [40]:
vec = CountVectorizer(ngram_range = (3, 3))
bow = vec.fit_transform(x_train)
clf = LogisticRegression(random_state = 42)
clf.fit(bow, y_train)
pred = clf.predict(vec.transform(x_test))
print(classification_report(pred, y_test))

              precision    recall  f1-score   support

    negative       0.47      0.72      0.57     18261
    positive       0.82      0.61      0.70     38448

    accuracy                           0.65     56709
   macro avg       0.64      0.67      0.63     56709
weighted avg       0.71      0.65      0.66     56709



(как вы думаете, почему в результатах теперь такой разброс по сравнению с униграммами?)

## TF-IDF векторизация

`TfidfVectorizer` делает то же, что и `CountVectorizer`, но в качестве значений – tf-idf каждого слова.

Как считается tf-idf:

TF (term frequency) – относительная частотность слова в документе:
$$ TF(t,d) = \frac{n_t}{\sum_k n_k} $$

`t` -- слово (term), `d` -- документ, $n_t$ -- количество вхождений слова, $n_k$ -- количество вхождений остальных слов

IDF (inverse document frequency) – обратная частота документов, в которых есть это слово:
$$ IDF(t, D) = \mbox{log} \frac{|D|}{|{d : t \in d}|} $$

`t` -- слово (term), `D` -- коллекция документов

Перемножаем их:
$$TFIDF_(t,d,D) = TF(t,d) \times IDF(i, D)$$

Сакральный смысл – если слово часто встречается в одном документе, но в целом по корпусу встречается в небольшом 
количестве документов, у него высокий TF-IDF.

In [41]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [42]:
vec = TfidfVectorizer(ngram_range=(1, 1))
bow = vec.fit_transform(x_train)
clf = LogisticRegression(random_state=42)
clf.fit(bow, y_train)
pred = clf.predict(vec.transform(x_test))
print(classification_report(pred, y_test))

              precision    recall  f1-score   support

    negative       0.73      0.77      0.75     26654
    positive       0.78      0.75      0.77     30055

    accuracy                           0.76     56709
   macro avg       0.76      0.76      0.76     56709
weighted avg       0.76      0.76      0.76     56709



В этот раз получилось хуже :( Вернёмся к `CountVectorizer`.

## Токенизация

Токенизировать -- значит, поделить текст на слова, или *токены*.

Самый наивный способ токенизировать текст -- разделить с помощью `split`. Но `split` упускает очень много всего, например, банально не отделяет пунктуацию от слов. Кроме этого, есть ещё много менее тривиальных проблем. Поэтому лучше использовать готовые токенизаторы.

In [43]:
from nltk.tokenize import word_tokenize

В nltk вообще есть довольно много токенизаторов:

In [44]:
from nltk import tokenize
dir(tokenize)[:16]

['BlanklineTokenizer',
 'LegalitySyllableTokenizer',
 'LineTokenizer',
 'MWETokenizer',
 'NLTKWordTokenizer',
 'PunktSentenceTokenizer',
 'PunktTokenizer',
 'RegexpTokenizer',
 'ReppTokenizer',
 'SExprTokenizer',
 'SpaceTokenizer',
 'StanfordSegmenter',
 'SyllableTokenizer',
 'TabTokenizer',
 'TextTilingTokenizer',
 'ToktokTokenizer']

Они умеют выдавать индексы начала и конца каждого токена:

In [46]:
help(tokenize.TreebankWordTokenizer())

Help on TreebankWordTokenizer in module nltk.tokenize.treebank object:

class TreebankWordTokenizer(nltk.tokenize.api.TokenizerI)
 |  The Treebank tokenizer uses regular expressions to tokenize text as in Penn Treebank.
 |  
 |  This tokenizer performs the following steps:
 |  
 |  - split standard contractions, e.g. ``don't`` -> ``do n't`` and ``they'll`` -> ``they 'll``
 |  - treat most punctuation characters as separate tokens
 |  - split off commas and single quotes, when followed by whitespace
 |  - separate periods that appear at the end of line
 |  
 |  >>> from nltk.tokenize import TreebankWordTokenizer
 |  >>> s = '''Good muffins cost $3.88\nin New York.  Please buy me\ntwo of them.\nThanks.'''
 |  >>> TreebankWordTokenizer().tokenize(s)
 |  ['Good', 'muffins', 'cost', '$', '3.88', 'in', 'New', 'York.', 'Please', 'buy', 'me', 'two', 'of', 'them.', 'Thanks', '.']
 |  >>> s = "They'll save and invest more."
 |  >>> TreebankWordTokenizer().tokenize(s)
 |  ['They', "'ll", 'save', 

In [47]:
tokenize.TreebankWordTokenizer().tokenize("don't stop me")

['do', "n't", 'stop', 'me']

А некоторые -- вообще не для текста на естественном языке (не очень понятно, зачем это в nltk :)):

In [48]:
tokenize.SExprTokenizer().tokenize("(a (b c)) d e (f)")

['(a (b c))', 'd', 'e', '(f)']

## Стоп-слова и пунктуация

*Стоп-слова* -- это слова, которые часто встречаются практически в любом тексте и ничего интересного не говорят о конретном документе, то есть играют роль шума. Поэтому их принято убирать. По той же причине убирают и пунктуацию.

In [49]:
# у вас здесь, вероятно, выскочит ошибка и надо будет загрузить стоп слова (в тексте ошибки написано, как)
from nltk.corpus import stopwords
print(stopwords.words('russian'))

['и', 'в', 'во', 'не', 'что', 'он', 'на', 'я', 'с', 'со', 'как', 'а', 'то', 'все', 'она', 'так', 'его', 'но', 'да', 'ты', 'к', 'у', 'же', 'вы', 'за', 'бы', 'по', 'только', 'ее', 'мне', 'было', 'вот', 'от', 'меня', 'еще', 'нет', 'о', 'из', 'ему', 'теперь', 'когда', 'даже', 'ну', 'вдруг', 'ли', 'если', 'уже', 'или', 'ни', 'быть', 'был', 'него', 'до', 'вас', 'нибудь', 'опять', 'уж', 'вам', 'ведь', 'там', 'потом', 'себя', 'ничего', 'ей', 'может', 'они', 'тут', 'где', 'есть', 'надо', 'ней', 'для', 'мы', 'тебя', 'их', 'чем', 'была', 'сам', 'чтоб', 'без', 'будто', 'чего', 'раз', 'тоже', 'себе', 'под', 'будет', 'ж', 'тогда', 'кто', 'этот', 'того', 'потому', 'этого', 'какой', 'совсем', 'ним', 'здесь', 'этом', 'один', 'почти', 'мой', 'тем', 'чтобы', 'нее', 'сейчас', 'были', 'куда', 'зачем', 'всех', 'никогда', 'можно', 'при', 'наконец', 'два', 'об', 'другой', 'хоть', 'после', 'над', 'больше', 'тот', 'через', 'эти', 'нас', 'про', 'всего', 'них', 'какая', 'много', 'разве', 'три', 'эту', 'моя', 'впр

In [50]:
print(stopwords.words('spanish'))

['de', 'la', 'que', 'el', 'en', 'y', 'a', 'los', 'del', 'se', 'las', 'por', 'un', 'para', 'con', 'no', 'una', 'su', 'al', 'lo', 'como', 'más', 'pero', 'sus', 'le', 'ya', 'o', 'este', 'sí', 'porque', 'esta', 'entre', 'cuando', 'muy', 'sin', 'sobre', 'también', 'me', 'hasta', 'hay', 'donde', 'quien', 'desde', 'todo', 'nos', 'durante', 'todos', 'uno', 'les', 'ni', 'contra', 'otros', 'ese', 'eso', 'ante', 'ellos', 'e', 'esto', 'mí', 'antes', 'algunos', 'qué', 'unos', 'yo', 'otro', 'otras', 'otra', 'él', 'tanto', 'esa', 'estos', 'mucho', 'quienes', 'nada', 'muchos', 'cual', 'poco', 'ella', 'estar', 'estas', 'algunas', 'algo', 'nosotros', 'mi', 'mis', 'tú', 'te', 'ti', 'tu', 'tus', 'ellas', 'nosotras', 'vosotros', 'vosotras', 'os', 'mío', 'mía', 'míos', 'mías', 'tuyo', 'tuya', 'tuyos', 'tuyas', 'suyo', 'suya', 'suyos', 'suyas', 'nuestro', 'nuestra', 'nuestros', 'nuestras', 'vuestro', 'vuestra', 'vuestros', 'vuestras', 'esos', 'esas', 'estoy', 'estás', 'está', 'estamos', 'estáis', 'están', 'e

In [51]:
print(stopwords.words('english'))

['i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', "you're", "you've", "you'll", "you'd", 'your', 'yours', 'yourself', 'yourselves', 'he', 'him', 'his', 'himself', 'she', "she's", 'her', 'hers', 'herself', 'it', "it's", 'its', 'itself', 'they', 'them', 'their', 'theirs', 'themselves', 'what', 'which', 'who', 'whom', 'this', 'that', "that'll", 'these', 'those', 'am', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'having', 'do', 'does', 'did', 'doing', 'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as', 'until', 'while', 'of', 'at', 'by', 'for', 'with', 'about', 'against', 'between', 'into', 'through', 'during', 'before', 'after', 'above', 'below', 'to', 'from', 'up', 'down', 'in', 'out', 'on', 'off', 'over', 'under', 'again', 'further', 'then', 'once', 'here', 'there', 'when', 'where', 'why', 'how', 'all', 'any', 'both', 'each', 'few', 'more', 'most', 'other', 'some', 'such', 'no', 'nor', 'not', 'only', 'own', 'same', 'so', 'than', '

In [52]:
from string import punctuation
punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [53]:
noise = stopwords.words('russian') + list(punctuation)

In [54]:
noise

['и',
 'в',
 'во',
 'не',
 'что',
 'он',
 'на',
 'я',
 'с',
 'со',
 'как',
 'а',
 'то',
 'все',
 'она',
 'так',
 'его',
 'но',
 'да',
 'ты',
 'к',
 'у',
 'же',
 'вы',
 'за',
 'бы',
 'по',
 'только',
 'ее',
 'мне',
 'было',
 'вот',
 'от',
 'меня',
 'еще',
 'нет',
 'о',
 'из',
 'ему',
 'теперь',
 'когда',
 'даже',
 'ну',
 'вдруг',
 'ли',
 'если',
 'уже',
 'или',
 'ни',
 'быть',
 'был',
 'него',
 'до',
 'вас',
 'нибудь',
 'опять',
 'уж',
 'вам',
 'ведь',
 'там',
 'потом',
 'себя',
 'ничего',
 'ей',
 'может',
 'они',
 'тут',
 'где',
 'есть',
 'надо',
 'ней',
 'для',
 'мы',
 'тебя',
 'их',
 'чем',
 'была',
 'сам',
 'чтоб',
 'без',
 'будто',
 'чего',
 'раз',
 'тоже',
 'себе',
 'под',
 'будет',
 'ж',
 'тогда',
 'кто',
 'этот',
 'того',
 'потому',
 'этого',
 'какой',
 'совсем',
 'ним',
 'здесь',
 'этом',
 'один',
 'почти',
 'мой',
 'тем',
 'чтобы',
 'нее',
 'сейчас',
 'были',
 'куда',
 'зачем',
 'всех',
 'никогда',
 'можно',
 'при',
 'наконец',
 'два',
 'об',
 'другой',
 'хоть',
 'после',
 'на

## Лемматизация

Лемматизация – это сведение разных форм одного слова к начальной форме – *лемме*. Почему это хорошо?
* Во-первых, мы хотим рассматривать как отдельную фичу каждое *слово*, а не каждую его отдельную форму.
* Во-вторых, некоторые стоп-слова стоят только в начальной форме, и без лематизации выкидываем мы только её.

Для русского есть два хороших лемматизатора: mystem и pymorphy:

### [Mystem](https://tech.yandex.ru/mystem/)
Как с ним работать:
* можно скачать mystem и запускать [из терминала с разными параметрами](https://yandex.ru/dev/mystem/?ysclid=m83e8gkero153283053)
* [pymystem3](https://pythonhosted.org/pymystem3/pymystem3.html) - обертка для питона, работает медленнее, но это удобно

In [ ]:
#!pip install pymystem3

In [55]:
from pymystem3 import Mystem
mystem_analyzer = Mystem()

Мы инициализировали Mystem c дефолтными параметрами. А вообще параметры есть такие:
* mystem_bin - путь к `mystem`, если их несколько
* grammar_info - нужна ли грамматическая информация или только леммы (по дефолту нужна)
* disambiguation - нужно ли снятие омонимии - дизамбигуация (по дефолту нужна)
* entire_input - нужно ли сохранять в выводе все (пробелы всякие, например), или можно выкинуть (по дефолту оставляется все)

Методы Mystem принимают строку, токенизатор вшит внутри. Можно, конечно, и пословно анализировать, но тогда он не сможет учитывать контекст.

Можно просто лемматизировать текст:

In [56]:
example = 'Lorem ipsum dolor sit amet, consectetuer adipiscing elit. Maecenas porttitor congue massa. Fusce posuere, magna sed pulvinar ultricies, purus lectus malesuada libero, sit amet commodo magna eros quis urna.\
            Nunc viverra imperdiet enim. Fusce est. Vivamus a tellus.\
            Pellentesque habitant morbi tristique senectus et netus et malesuada fames ac turpis egestas. Proin pharetra nonummy pede. Mauris et orci.'

In [57]:
example

'Lorem ipsum dolor sit amet, consectetuer adipiscing elit. Maecenas porttitor congue massa. Fusce posuere, magna sed pulvinar ultricies, purus lectus malesuada libero, sit amet commodo magna eros quis urna.            Nunc viverra imperdiet enim. Fusce est. Vivamus a tellus.            Pellentesque habitant morbi tristique senectus et netus et malesuada fames ac turpis egestas. Proin pharetra nonummy pede. Mauris et orci.'

In [58]:
print(mystem_analyzer.lemmatize(example))

['Lorem', ' ', 'ipsum', ' ', 'dolor', ' ', 'sit', ' ', 'amet', ', ', 'consectetuer', ' ', 'adipiscing', ' ', 'elit', '. ', 'Maecenas', ' ', 'porttitor', ' ', 'congue', ' ', 'massa', '. ', 'Fusce', ' ', 'posuere', ', ', 'magna', ' ', 'sed', ' ', 'pulvinar', ' ', 'ultricies', ', ', 'purus', ' ', 'lectus', ' ', 'malesuada', ' ', 'libero', ', ', 'sit', ' ', 'amet', ' ', 'commodo', ' ', 'magna', ' ', 'eros', ' ', 'quis', ' ', 'urna', '.            ', 'Nunc', ' ', 'viverra', ' ', 'imperdiet', ' ', 'enim', '. ', 'Fusce', ' ', 'est', '. ', 'Vivamus', ' ', 'a', ' ', 'tellus', '.            ', 'Pellentesque', ' ', 'habitant', ' ', 'morbi', ' ', 'tristique', ' ', 'senectus', ' ', 'et', ' ', 'netus', ' ', 'et', ' ', 'malesuada', ' ', 'fames', ' ', 'ac', ' ', 'turpis', ' ', 'egestas', '. ', 'Proin', ' ', 'pharetra', ' ', 'nonummy', ' ', 'pede', '. ', 'Mauris', ' ', 'et', ' ', 'orci', '.', '\n']


In [59]:
example = 'Для чего используются регулярные выражения для определения нужного формата, например телефонного номера или email-адреса;\
    для разбивки строк на подстроки;\
    для поиска, замены и извлечения символов;\
    для быстрого выполнения нетривиальных операций.'

In [61]:
print(mystem_analyzer.lemmatize(example))

['для', ' ', 'что', ' ', 'использоваться', ' ', 'регулярный', ' ', 'выражение', ' ', 'для', ' ', 'определение', ' ', 'нужный', ' ', 'формат', ', ', 'например', ' ', 'телефонный', ' ', 'номер', ' ', 'или', ' ', 'email', '-', 'адрес', ';    ', 'для', ' ', 'разбивка', ' ', 'строка', ' ', 'на', ' ', 'подстрока', ';    ', 'для', ' ', 'поиск', ', ', 'замена', ' ', 'и', ' ', 'извлечение', ' ', 'символ', ';    ', 'для', ' ', 'быстрый', ' ', 'выполнение', ' ', 'нетривиальный', ' ', 'операция', '.', '\n']


А можно получить грамматическую информацию:

In [62]:
mystem_analyzer.analyze(example)

[{'analysis': [{'lex': 'для', 'wt': 1, 'gr': 'PR='}], 'text': 'Для'},
 {'text': ' '},
 {'analysis': [{'lex': 'что',
    'wt': 0.8207349868,
    'gr': 'SPRO,ед,сред,неод=род'}],
  'text': 'чего'},
 {'text': ' '},
 {'analysis': [{'lex': 'использоваться',
    'wt': 1,
    'gr': 'V,несов,нп=непрош,мн,изъяв,3-л'}],
  'text': 'используются'},
 {'text': ' '},
 {'analysis': [{'lex': 'регулярный',
    'wt': 1,
    'gr': 'A=(вин,мн,полн,неод|им,мн,полн)'}],
  'text': 'регулярные'},
 {'text': ' '},
 {'analysis': [{'lex': 'выражение',
    'wt': 1,
    'gr': 'S,сред,неод=(вин,мн|род,ед|им,мн)'}],
  'text': 'выражения'},
 {'text': ' '},
 {'analysis': [{'lex': 'для', 'wt': 1, 'gr': 'PR='}], 'text': 'для'},
 {'text': ' '},
 {'analysis': [{'lex': 'определение',
    'wt': 1,
    'gr': 'S,сред,неод=(вин,мн|род,ед|им,мн)'}],
  'text': 'определения'},
 {'text': ' '},
 {'analysis': [{'lex': 'нужный',
    'wt': 1,
    'gr': 'A=(вин,ед,полн,муж,од|род,ед,полн,муж|род,ед,полн,сред)'}],
  'text': 'нужного'},
 {

In [63]:
import re
def my_preproc(text):
    text = re.sub('[{}]'.format(punctuation), '', text)
    text = mystem_analyzer.lemmatize(text)
    return [word for word in text if word not in stopwords.words('russian') + [' ', '\n']]

In [64]:
my_preproc(example)

['использоваться',
 'регулярный',
 'выражение',
 'определение',
 'нужный',
 'формат',
 'например',
 'телефонный',
 'номер',
 'emailадреса',
 '    ',
 'разбивка',
 'строка',
 'подстрока',
 '    ',
 'поиск',
 'замена',
 'извлечение',
 'символ',
 '    ',
 'быстрый',
 'выполнение',
 'нетривиальный',
 'операция']

# Word2Vec


In [65]:
from gensim.test.utils import common_texts #предоставляет инструменты для анализа текстовых данных, включая создание векторных представлений слов, моделирование тем и другие методы обработки текста
from gensim.models import Word2Vec

In [66]:
help(common_texts)

Help on list object:

class list(object)
 |  list(iterable=(), /)
 |  
 |  Built-in mutable sequence.
 |  
 |  If no argument is given, the constructor creates a new empty list.
 |  The argument must be an iterable if specified.
 |  
 |  Methods defined here:
 |  
 |  __add__(self, value, /)
 |      Return self+value.
 |  
 |  __contains__(self, key, /)
 |      Return key in self.
 |  
 |  __delitem__(self, key, /)
 |      Delete self[key].
 |  
 |  __eq__(self, value, /)
 |      Return self==value.
 |  
 |  __ge__(self, value, /)
 |      Return self>=value.
 |  
 |  __getattribute__(self, name, /)
 |      Return getattr(self, name).
 |  
 |  __getitem__(...)
 |      x.__getitem__(y) <==> x[y]
 |  
 |  __gt__(self, value, /)
 |      Return self>value.
 |  
 |  __iadd__(self, value, /)
 |      Implement self+=value.
 |  
 |  __imul__(self, value, /)
 |      Implement self*=value.
 |  
 |  __init__(self, /, *args, **kwargs)
 |      Initialize self.  See help(type(self)) for accurate sign

In [67]:
common_texts

[['human', 'interface', 'computer'],
 ['survey', 'user', 'computer', 'system', 'response', 'time'],
 ['eps', 'user', 'interface', 'system'],
 ['system', 'human', 'system', 'eps'],
 ['user', 'response', 'time'],
 ['trees'],
 ['graph', 'trees'],
 ['graph', 'minors', 'trees'],
 ['graph', 'minors', 'survey']]

In [68]:
skipgram = 1 # Этот параметр указывает, что мы хотим использовать модель Skip-gram для обучения векторных представлений слов. Если бы это значение было 0, использовалась бы модель Continuous Bag of Words (CBOW).
negative_sampling = 5 #Этот параметр задает количество отрицательных примеров, которые будут использоваться для обучения модели. В данном случае для каждого положительного примера будет 5 отрицательных.
w2v_model = Word2Vec(sentences = common_texts, #Модель будет обучена на наборе данных common_texts, который мы импортировали ранее из библиотеки Gensim
                     vector_size = 100, #Указывает размерность векторных представлений слов. В данном случае каждое слово будет представлено вектором размером 100
                     window = 5, #Определяет размер окна контекста. Это означает, что при обучении модели будут учитываться 5 слов слева и 5 слов справа от целевого слова
                     min_count = 1, #Этот параметр указывает, что слова, встречающиеся реже 1 раза, будут игнорироваться. В данном случае все слова будут включены в модель, так как минимальная частота равна 1
                     workers = 4, #Определяет количество потоков, которые будут использоваться для обучения модели. В данном случае модель будет использовать 4 потока для ускорения процесса обучения
                     sg = skipgram, #Указывает, какую модель использовать для обучения. Если skipgram равно 1, будет использоваться модель Skip-gram, если 0 — модель CBOW.
                     negative = negative_sampling) #Указывает количество отрицательных выборок для каждого положительного примера

w2v_model.save("word2vec.model") #Этот метод сохраняет обученную модель в файл с именем "word2vec.model". Это позволяет впоследствии загрузить модель и использовать её без повторного обучения

In [70]:
vector = w2v_model.wv['computer']  # Здесь мы используем модель w2v_model, чтобы получить вектор, представляющий слово "computer".
vector

array([-0.00515774, -0.00667028, -0.0077791 ,  0.00831315, -0.00198292,
       -0.00685696, -0.0041556 ,  0.00514562, -0.00286997, -0.00375075,
        0.0016219 , -0.0027771 , -0.00158482,  0.0010748 , -0.00297881,
        0.00852176,  0.00391207, -0.00996176,  0.00626142, -0.00675622,
        0.00076966,  0.00440552, -0.00510486, -0.00211128,  0.00809783,
       -0.00424503, -0.00763848,  0.00926061, -0.00215612, -0.00472081,
        0.00857329,  0.00428459,  0.0043261 ,  0.00928722, -0.00845554,
        0.00525685,  0.00203994,  0.0041895 ,  0.00169839,  0.00446543,
        0.0044876 ,  0.0061063 , -0.00320303, -0.00457706, -0.00042664,
        0.00253447, -0.00326412,  0.00605948,  0.00415534,  0.00776685,
        0.00257002,  0.00811905, -0.00138761,  0.00808028,  0.0037181 ,
       -0.00804967, -0.00393476, -0.0024726 ,  0.00489447, -0.00087241,
       -0.00283173,  0.00783599,  0.00932561, -0.0016154 , -0.00516075,
       -0.00470313, -0.00484746, -0.00960562,  0.00137242, -0.00

In [71]:
vector.shape

(100,)

In [72]:
w2v_model.wv['human']

array([ 9.7702928e-03,  8.1651136e-03,  1.2809718e-03,  5.0975787e-03,
        1.4081288e-03, -6.4551616e-03, -1.4280510e-03,  6.4491653e-03,
       -4.6173059e-03, -3.9930656e-03,  4.9244044e-03,  2.7130984e-03,
       -1.8479753e-03, -2.8769434e-03,  6.0107317e-03, -5.7167388e-03,
       -3.2367026e-03, -6.4878250e-03, -4.2346325e-03, -8.5809948e-03,
       -4.4697891e-03, -8.5112294e-03,  1.4037776e-03, -8.6181965e-03,
       -9.9166557e-03, -8.2016252e-03, -6.7726658e-03,  6.6805850e-03,
        3.7845564e-03,  3.5616636e-04, -2.9579818e-03, -7.4283206e-03,
        5.3341867e-04,  4.9989222e-04,  1.9561886e-04,  8.5259555e-04,
        7.8633073e-04, -6.8160298e-05, -8.0070542e-03, -5.8702733e-03,
       -8.3829118e-03, -1.3120425e-03,  1.8206370e-03,  7.4171280e-03,
       -1.9634271e-03, -2.3252917e-03,  9.4871549e-03,  7.9704521e-05,
       -2.4045217e-03,  8.6048469e-03,  2.6870037e-03, -5.3439722e-03,
        6.5881060e-03,  4.5101536e-03, -7.0544672e-03, -3.2317400e-04,
      

In [73]:
sims = w2v_model.wv.most_similar('computer', topn = 10)  # Метод most_similar возвращает список кортежей, каждый из которых содержит слово и его схожесть (или косинусное расстояние) с заданным словом. Параметр topn=10 указывает, что мы хотим получить 10 наиболее похожих слов
sims

[('system', 0.21617141366004944),
 ('survey', 0.04468922317028046),
 ('interface', 0.015203384682536125),
 ('time', 0.0019510771380737424),
 ('trees', -0.03284314274787903),
 ('human', -0.07424270361661911),
 ('response', -0.09324456751346588),
 ('graph', -0.09575342386960983),
 ('eps', -0.10513808578252792),
 ('user', -0.1690933257341385)]

In [74]:
w2v_model

In [75]:
w2v_model.wv['human'].shape #используется для получения формы (размерности) вектора, представляющего слово "human" в модели Word2Vec

(100,)

In [76]:
w2v_model.wv['human']

array([ 9.7702928e-03,  8.1651136e-03,  1.2809718e-03,  5.0975787e-03,
        1.4081288e-03, -6.4551616e-03, -1.4280510e-03,  6.4491653e-03,
       -4.6173059e-03, -3.9930656e-03,  4.9244044e-03,  2.7130984e-03,
       -1.8479753e-03, -2.8769434e-03,  6.0107317e-03, -5.7167388e-03,
       -3.2367026e-03, -6.4878250e-03, -4.2346325e-03, -8.5809948e-03,
       -4.4697891e-03, -8.5112294e-03,  1.4037776e-03, -8.6181965e-03,
       -9.9166557e-03, -8.2016252e-03, -6.7726658e-03,  6.6805850e-03,
        3.7845564e-03,  3.5616636e-04, -2.9579818e-03, -7.4283206e-03,
        5.3341867e-04,  4.9989222e-04,  1.9561886e-04,  8.5259555e-04,
        7.8633073e-04, -6.8160298e-05, -8.0070542e-03, -5.8702733e-03,
       -8.3829118e-03, -1.3120425e-03,  1.8206370e-03,  7.4171280e-03,
       -1.9634271e-03, -2.3252917e-03,  9.4871549e-03,  7.9704521e-05,
       -2.4045217e-03,  8.6048469e-03,  2.6870037e-03, -5.3439722e-03,
        6.5881060e-03,  4.5101536e-03, -7.0544672e-03, -3.2317400e-04,
      

In [77]:
result = w2v_model.wv.most_similar(positive = ['human', 'system'], negative = ['computer'], topn = 1) 

'''
используется метод most_similar для получения слов, 
которые наиболее похожи на комбинацию слов "human" и "system", 
при этом исключая слово "computer".

positive=['human', 'system']: Это список слов, которые мы хотим учесть при поиске схожих слов. В данном случае мы ищем слова, которые похожи на оба слова "human" и "system".
negative=['computer']: Это слово, которое мы хотим исключить из результатов. То есть, 
мы ищем слова, которые похожи на "human" и "system", но не похожи на "computer".
topn = 1: Указывает, что мы хотим получить только одно слово, наиболее похожее на заданную комбинацию.
'''

print(result)

[('response', 0.17535406351089478)]


In [78]:
w2v_model.wv.most_similar(positive = ['human', 'system'], negative = ['computer'], topn = 5)

[('response', 0.17535406351089478),
 ('eps', 0.125144824385643),
 ('interface', 0.09263758361339569),
 ('trees', 0.09036029130220413),
 ('minors', 0.08231795579195023)]

In [79]:
result = w2v_model.wv.most_similar(positive = ['human', 'system'], negative = ['trees'], topn = 1) 
print(result)

[('interface', 0.16055169701576233)]


In [80]:
w2v_model.wv.most_similar(positive = ['trees'], negative = ['survey'], topn = 5)

[('minors', 0.1889941543340683),
 ('response', 0.11202939599752426),
 ('time', 0.10834681987762451),
 ('human', 0.10668007284402847),
 ('eps', 0.00276385061442852)]

### Предобученные эмбеддинги

In [ ]:
#!wget http://nlp.stanford.edu/data/glove.6B.zip

In [ ]:
#!unzip glove.6B.zip

In [81]:
from gensim.scripts.glove2word2vec import glove2word2vec
glove2word2vec('glove.6B.100d.txt', 'glove.6B.100d.txt.word2vec') #для преобразования модели GloVe (Global Vectors for Word Representation) в формат, совместимый с библиотекой Gensim, который позволяет загружать и использовать векторные представления слов. 

/var/folders/7t/_ywngh7s4vzflssy506tj1m40000gn/T/ipykernel_70721/1259735528.py:2: DeprecationWarning: Call to deprecated `glove2word2vec` (KeyedVectors.load_word2vec_format(.., binary=False, no_header=True) loads GLoVE text vectors.).
  glove2word2vec('glove.6B.100d.txt', 'glove.6B.100d.txt.word2vec') #для преобразования модели GloVe (Global Vectors for Word Representation) в формат, совместимый с библиотекой Gensim, который позволяет загружать и использовать векторные представления слов.


(400000, 100)

In [82]:
from gensim.models import KeyedVectors
# load the Stanford GloVe model
filename = 'glove.6B.100d.txt.word2vec' #путь к файлу, в котором сохранены векторные представления слов в формате Word2Vec.
glove_model = KeyedVectors.load_word2vec_format(filename, binary = False) #Загружает модель из указанного файла. Параметр binary=False указывает, что файл не в двоичном формате, а в текстовом.
# calculate: (king - man) + woman = ?
result = glove_model.most_similar(positive = ['woman', 'king'], negative = ['man'], topn = 1)

'''
Вычисляем вектор, который получается из выражения (king−man)+woman.
positive=['woman', 'king']: Это список слов, которые мы хотим учесть при вычислении. Мы ищем слова, которые семантически похожи на "woman" и "king".
negative=['man']: Это слово, которое мы хотим исключить из результата. Мы ищем слова, которые похожи на "king" и "woman", но не похожи на "man".
topn=1: Указывает, что мы хотим получить только одно слово, наиболее похожее на результат.
'''

print(result)

[('queen', 0.7698541283607483)]


In [83]:
glove_model.most_similar(positive = ['woman', 'king'], negative = ['man'], topn = 1)

[('queen', 0.7698541283607483)]

In [84]:
glove_model.most_similar(positive = ['car', 'bicycle'], topn = 1)

[('motorcycle', 0.8456250429153442)]

In [85]:
glove_model.most_similar(positive = ['car', 'bicycle'], negative = ['motorcycle'], topn = 1)

[('bus', 0.7515897154808044)]

In [86]:
glove_model.most_similar(positive = ['woman', 'king'], 
                         negative = ['crown'], topn = 1)

[('mother', 0.7773604989051819)]

In [87]:
glove_model.most_similar(positive = ['woman', 'cat'], 
                         negative = ['animal'], topn = 1)

[('girl', 0.7711177468299866)]

In [88]:
glove_model.most_similar(positive = ['dog', 'cat'], 
                         negative = ['animal'], topn = 1)

[('puppy', 0.6911072731018066)]

In [89]:
glove_model.most_similar(positive = ['house', 'trees'], 
                         negative = ['human'], topn = 1)

[('bushes', 0.6633507013320923)]

In [91]:
glove_model.most_similar(positive = ['rock', 'guitar'], topn = 1)

[('band', 0.8304225206375122)]

In [92]:
glove_model.most_similar(positive = ['rock', 'guitar'], 
                         negative = ['music', 'woman'], topn = 1)

[('guitars', 0.5721947550773621)]

In [93]:
glove_model.most_similar(positive = ['rock', 'guitar', 'woman'], 
                         negative = ['music'], topn = 1)

[('man', 0.7345726490020752)]

In [94]:
glove_model.most_similar(positive = ['human', 'moon'], topn = 1)

[('earth', 0.7299715876579285)]

In [95]:
glove_model.most_similar(positive = ['math', 'humanity'], topn = 1)

[('learning', 0.6605393290519714)]

In [96]:
glove_model.most_similar(positive = ['professor', 'juice'], 
                         negative = ['sausage'], topn = 1)

[('university', 0.5892939567565918)]